<a href="https://colab.research.google.com/github/silverwing-coder/AI/blob/master/GoogleColabs/TextGeneration_Summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h2>Programming Environment ....</h2>
<h3>Authentication (TOKEN submission) </h3>
<ol>
  <li>Save to token in "secrets"</li>
  <li>Use access token </li>

  ```py
    import os
    from google.colab import userdata
    userdata.get('HF_TOKEN')
  ```
</ol>

<h3>Performance improvement </h3>
<ol>
  <li>Switch to GPU mode to save execution time: "Runtime >> Change runtime type"</li>
  <li>(Optional) Move your model and inputs to the GPU </li>

  ```py
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    output = model.generate(**inputs, max_new_tokens=100)
  ```
  <li>(Optional) Load the model in lower precision: Quantization </li>

  ```py
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(load_in_4bit=True)
    model = AutoModelForCausalLM.from_pretrained(
      model="your-model-id",
      quantization_config=quantization_config
    )
  ```
</ol>

In [ ]:
! pip install transformers
! pip install bitsandbytes    # quantization
! pip install accelerate      # quantization

In [ ]:
from transformers import pipeline
import torch
from transformers import BitsAndBytesConfig

import os
from google.colab import userdata
userdata.get('HF_TOKEN')

# from google.colab import files
# upload = files.upload()

with open('GenAITest/test.txt', 'r') as f:
  text = f.read()

# text = "Three people were dead at the scene and another was taken to a hospital in unknown condition, the department in a statement. \
# At least one of the deceased appeared to be a pedestrian on the ground when the crash happened. Fire Capt. Branden Silverman said at the scene.\
# The helicopter crash occurred after two people were killed and others were injured in a crash between a Los Angeles Metro bus \
# and a passenger vehicle on Nordhoff Street, according to the fire department.It said that the bus crash happened at 5:03 p.m. \
# and that firefighters worked to free the trapped people and assess the bus passengers."

model_id = "Qwen/Qwen3-4B-Instruct-2507"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
    # bnb_4bit_quant_type="nf4"
)

model = pipeline(
    task="text-generation",
    model=model_id,
    model_kwargs={
        "quantization_config": quantization_config,
        # "torch_dtype": torch.float16, torch_type was depricated
        "dtype": torch.float16,
        "device_map": "auto"    # Automatically maps layers to GPU
    }
)

messages = [
    {
        "role": "user",
        "content": "Summarize the following text in 3 bullet points:\n\n" + text
    }
]

out = model(messages, max_new_tokens=200)
print(out[0]["generated_text"][-1]["content"])


